# Sistemas de amortización

In [32]:
import pandas as pd 
import numpy as np

In [33]:
datos= pd.read_excel("datos/datos_limpio_automovil.xlsx")

In [34]:
datos.head()

,Unnamed: 0,ID,Edad,Ingresos,Monto_Inicial,Scoring_Crediticio,Meses_Empleo,Num_Creditos,Ratio_Interes,Duracion,...,Estudios_Escolar,Estudios_Grado Universitario,Estudios_Máster,Tipo_Jornada_Laboral_Autónomo,Tipo_Jornada_Laboral_Desempleado,Tipo_Jornada_Laboral_Jornada completa,Tipo_Jornada_Laboral_Tiempo parcial,Estado_Civil_Casado,Estado_Civil_Divorciado,Estado_Civil_Soltero
0,0,O77MPJ,26,19153,6973,706,9,1,19.05,36,...,False,True,False,False,False,True,False,False,False,True
1,8,IYOL1S,29,39531,40000,826,0,1,9.74,24,...,False,False,False,False,False,True,False,True,False,False
2,9,FRMZVN,29,40348,16448,740,9,1,14.73,24,...,False,False,True,False,False,True,False,False,False,True
3,26,YKHXVY,38,41732,42040,661,173,2,13.24,48,...,False,True,False,False,False,False,True,True,False,False
4,32,NHDJTK,35,35085,16406,759,181,1,9.74,36,...,True,False,False,False,False,True,False,True,False,False


In [35]:
datos.columns


Index(['Unnamed: 0', 'ID', 'Edad', 'Ingresos', 'Monto_Inicial',
       'Scoring_Crediticio', 'Meses_Empleo', 'Num_Creditos', 'Ratio_Interes',
       'Duracion', 'Ratio_Deuda_Ingresos', 'Posesion_Hipoteca',
       'Personas_Cargo', 'Fiador', 'Impago', 'Prima', 'Estudios_Doctorado',
       'Estudios_Escolar', 'Estudios_Grado Universitario', 'Estudios_Máster',
       'Tipo_Jornada_Laboral_Autónomo', 'Tipo_Jornada_Laboral_Desempleado',
       'Tipo_Jornada_Laboral_Jornada completa',
       'Tipo_Jornada_Laboral_Tiempo parcial', 'Estado_Civil_Casado',
       'Estado_Civil_Divorciado', 'Estado_Civil_Soltero'],
      dtype='str')

In [36]:
datos.shape

(76924, 27)

In [37]:
datos.info()

<class 'pandas.DataFrame'>
RangeIndex: 76924 entries, 0 to 76923
Data columns (total 27 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   Unnamed: 0                             76924 non-null  int64  
 1   ID                                     76924 non-null  str    
 2   Edad                                   76924 non-null  int64  
 3   Ingresos                               76924 non-null  int64  
 4   Monto_Inicial                          76924 non-null  int64  
 5   Scoring_Crediticio                     76924 non-null  int64  
 6   Meses_Empleo                           76924 non-null  int64  
 7   Num_Creditos                           76924 non-null  int64  
 8   Ratio_Interes                          76924 non-null  float64
 9   Duracion                               76924 non-null  int64  
 10  Ratio_Deuda_Ingresos                   76924 non-null  float64
 11  Posesion_Hipo

# 1- Cuadros de amortización 

Cambiamos los nombres para hacer mas facil la formula 
- Monto inicial-->capital_prestado
- Ratio Interes -->  interés_anual.
- Duración --> num_meses

In [38]:
datos = datos.rename(columns={
    "Monto_Inicial": "capital_prestado",
    "Ratio_Interes": "interes_anual",
    "Duracion": "num_meses"
})
print(datos.info())

<class 'pandas.DataFrame'>
RangeIndex: 76924 entries, 0 to 76923
Data columns (total 27 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   Unnamed: 0                             76924 non-null  int64  
 1   ID                                     76924 non-null  str    
 2   Edad                                   76924 non-null  int64  
 3   Ingresos                               76924 non-null  int64  
 4   capital_prestado                       76924 non-null  int64  
 5   Scoring_Crediticio                     76924 non-null  int64  
 6   Meses_Empleo                           76924 non-null  int64  
 7   Num_Creditos                           76924 non-null  int64  
 8   interes_anual                          76924 non-null  float64
 9   num_meses                              76924 non-null  int64  
 10  Ratio_Deuda_Ingresos                   76924 non-null  float64
 11  Posesion_Hipo

# Sistema Francés

La cuota es constante.
Los intereses disminuyen.
La amortización de capital aumenta.

Crear la función del sistema francés

In [39]:
def cuadro_frances(capital, interes_anual, meses):

    # Pasar el interés anual (%) a interés mensual decimal
    i = (interes_anual / 100) / 12

    # Cuota constante
    cuota = capital * (i * (1 + i)**meses) / ((1 + i)**meses - 1)

    saldo = capital

    tabla = []

    for periodo in range(1, meses + 1):

        intereses = saldo * i

        capital_amortizado = cuota - intereses

        saldo = saldo - capital_amortizado

        tabla.append([
            periodo,
            round(cuota, 2),
            round(intereses, 2),
            round(capital_amortizado, 2),
            round(max(0, saldo), 2)
        ])

    return pd.DataFrame(
        tabla,
        columns=[
            "Periodo",
            "Cuota",
            "Intereses",
            "Capital_Amortizado",
            "Saldo_Pendiente"
        ]
    )

Probamos que funciona con un prestamo

# Sistema Alemán

La amortización es constante.

Funcion del sistema alemán

In [40]:
def cuadro_aleman(capital, interes_anual, meses):

    i = (interes_anual / 100) / 12

    amortizacion_constante = capital / meses

    saldo = capital

    tabla = []

    for periodo in range(1, meses + 1):

        intereses = saldo * i

        cuota = amortizacion_constante + intereses

        saldo = saldo - amortizacion_constante

        tabla.append([
            periodo,
            round(cuota, 2),
            round(intereses, 2),
            round(amortizacion_constante, 2),
            round(max(0, saldo), 2)
        ])

    return pd.DataFrame(
        tabla,
        columns=[
            "Periodo",
            "Cuota",
            "Intereses",
            "Capital_Amortizado",
            "Saldo_Pendiente"
        ]
    )

Probarlo con el primer préstamo

In [41]:
capital = datos.loc[0, "capital_prestado"]
interes = datos.loc[0, "interes_anual"]
meses = datos.loc[0, "num_meses"]

frances = cuadro_frances(capital, interes, meses)
aleman = cuadro_aleman(capital, interes, meses)

print("CUADRO FRANCÉS")
print(frances.head())

print("\nCUADRO ALEMÁN")
print(aleman.head())

CUADRO FRANCÉS
   Periodo   Cuota  Intereses  Capital_Amortizado  Saldo_Pendiente
0        1  255.78     110.70              145.08          6827.92
1        2  255.78     108.39              147.39          6680.53
2        3  255.78     106.05              149.73          6530.81
3        4  255.78     103.68              152.10          6378.70
4        5  255.78     101.26              154.52          6224.19

CUADRO ALEMÁN
   Periodo   Cuota  Intereses  Capital_Amortizado  Saldo_Pendiente
0        1  304.39     110.70              193.69          6779.31
1        2  301.32     107.62              193.69          6585.61
2        3  298.24     104.55              193.69          6391.92
3        4  295.17     101.47              193.69          6198.22
4        5  292.09      98.40              193.69          6004.53


Cuadro frances para todos los prestamos

In [42]:
cuadros_frances = {}

for _, fila in datos.iterrows():

    cuadros_frances[fila["ID"]] = cuadro_frances(
        fila["capital_prestado"],
        fila["interes_anual"],
        fila["num_meses"]
    )

Cuadro aleman para todos los prestamos

In [43]:
cuadros_aleman = {}

for _, fila in datos.iterrows():

    cuadros_aleman[fila["ID"]] = cuadro_aleman(
        fila["capital_prestado"],
        fila["interes_anual"],
        fila["num_meses"]
    )

Ver el cuadro de un préstamo concreto

In [44]:
id_prestamo = datos.loc[0, "ID"]

print(cuadros_frances[id_prestamo])
print(cuadros_aleman[id_prestamo])
print(cuadros_aleman[id_prestamo])


    Periodo   Cuota  Intereses  Capital_Amortizado  Saldo_Pendiente
0         1  255.78     110.70              145.08          6827.92
1         2  255.78     108.39              147.39          6680.53
2         3  255.78     106.05              149.73          6530.81
3         4  255.78     103.68              152.10          6378.70
4         5  255.78     101.26              154.52          6224.19
5         6  255.78      98.81              156.97          6067.22
6         7  255.78      96.32              159.46          5907.76
7         8  255.78      93.79              161.99          5745.76
8         9  255.78      91.21              164.56          5581.20
9        10  255.78      88.60              167.18          5414.02
10       11  255.78      85.95              169.83          5244.19
11       12  255.78      83.25              172.53          5071.66
12       13  255.78      80.51              175.27          4896.40
13       14  255.78      77.73              178.

Comparar francés y alemán para cada préstamo

In [ ]:
comparacion = []

for _, fila in datos.iterrows():

    id_prestamo = fila["ID"]

    cuadro_f = cuadro_frances(
        fila["capital_prestado"],
        fila["interes_anual"],
        fila["num_meses"]
    )

    cuadro_a = cuadro_aleman(
        fila["capital_prestado"],
        fila["interes_anual"],
        fila["num_meses"]
    )

    intereses_frances = cuadro_f["Intereses"].sum()
    intereses_aleman = cuadro_a["Intereses"].sum()

    mejor = "Francés"

    if intereses_aleman < intereses_frances:
        mejor = "Alemán"

    comparacion.append([
        id_prestamo,
        round(intereses_frances, 2),
        round(intereses_aleman, 2),
        mejor
    ])

comparacion = pd.DataFrame(
    comparacion,
    columns=[
        "ID",
        "Intereses_Frances",
        "Intereses_Aleman",
        "Metodo_Recomendado"
    ]
)

comparacion.head()


,ID,Intereses_Frances,Intereses_Aleman,Metodo_Recomendado
0,O77MPJ,2235.01,2047.86,Alemán
1,IYOL1S,4184.01,4058.34,Alemán
2,FRMZVN,2641.59,2523.72,Alemán
3,YKHXVY,12336.41,11364.08,Alemán
4,NHDJTK,2579.50,2463.50,Alemán
